# Categorical Manipulation

### Loading Libraries

In [68]:
# Numerical Computing
import numpy as np

# Data Manipulation
import pandas as pd

# Data Visualisation
import seaborn as sns
import matplotlib.pyplot as plt

# OS
import requests
from io import StringIO



#### Retrieving Data

In [2]:
pd.set_option('display.max_rows', 10) 

In [3]:
# url = 'https://github.com/mattharrison/datasets/raw/master/data/'\
#     'siena2018-pres.csv'

In [4]:
url = 'https://github.com/mattharrison/datasets/raw/master/data/' \
      'vehicles.csv.zip'

In [5]:
df = pd.read_csv(url, engine='pyarrow', dtype_backend='pyarrow')

In [6]:
make = df.make

In [7]:
make

0        Alfa Romeo
1           Ferrari
2             Dodge
3             Dodge
4            Subaru
            ...    
41139        Subaru
41140        Subaru
41141        Subaru
41142        Subaru
41143        Subaru
Name: make, Length: 41144, dtype: string[pyarrow]

#### Frequency Counts

In [8]:
make.value_counts()

make
Chevrolet                           4003
Ford                                3371
Dodge                               2583
GMC                                 2494
Toyota                              2071
                                    ... 
Grumman Allied Industries              1
Environmental Rsch and Devp Corp       1
General Motors                         1
Goldacre                               1
Isis Imports Ltd                       1
Name: count, Length: 136, dtype: int64[pyarrow]

In [9]:
make.shape, make.nunique()

((41144,), 136)

#### Beneficts of Categories

In [10]:
cat_make = make.astype('category')

In [11]:
make.memory_usage(deep=True)

425767

In [12]:
cat_make.memory_usage(deep=True)

84533

In [13]:
%%timeit
cat_make.str.upper()

334 μs ± 2.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [14]:
%%timeit
make.str.upper()

414 μs ± 2.01 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [15]:
old_make = make.astype(str)

In [16]:
%%timeit
old_make.str.upper()

428 μs ± 3.47 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


#### Conversion to Ordinal Categories

In [17]:
make_type = pd.CategoricalDtype(
    categories=sorted(make.unique()), ordered=True)

In [18]:
ordered_make = make.astype(make_type)

ordered_make

0        Alfa Romeo
1           Ferrari
2             Dodge
3             Dodge
4            Subaru
            ...    
41139        Subaru
41140        Subaru
41141        Subaru
41142        Subaru
41143        Subaru
Name: make, Length: 41144, dtype: category
Categories (136, str): ['AM General' < 'ASC Incorporated' < 'Acura' < 'Alfa Romeo' ... 'Volvo' < 'Wallace Environmental' < 'Yugo' < 'smart']

In [19]:
(make
    .astype('category')
    .cat.as_ordered()
)

0        Alfa Romeo
1           Ferrari
2             Dodge
3             Dodge
4            Subaru
            ...    
41139        Subaru
41140        Subaru
41141        Subaru
41142        Subaru
41143        Subaru
Name: make, Length: 41144, dtype: category
Categories (136, string[pyarrow]): [AM General < ASC Incorporated < Acura < Alfa Romeo ... Volvo < Wallace Environmental < Yugo < smart]

In [20]:
ordered_make.max()

'smart'

In [21]:
cat_make.max()

TypeError: Categorical is not ordered for operation max
you can use .as_ordered() to change the Categorical to an ordered one


In [22]:
ordered_make.sort_values()

20288    AM General
20289    AM General
369      AM General
358      AM General
19314    AM General
            ...    
31289         smart
31290         smart
29605         smart
22974         smart
26882         smart
Name: make, Length: 41144, dtype: category
Categories (136, str): ['AM General' < 'ASC Incorporated' < 'Acura' < 'Alfa Romeo' ... 'Volvo' < 'Wallace Environmental' < 'Yugo' < 'smart']

#### The `.cat` Accesor

In [23]:
cat_make.cat.rename_categories(
    [c.lower() for c in cat_make.cat.categories])

0        alfa romeo
1           ferrari
2             dodge
3             dodge
4            subaru
            ...    
41139        subaru
41140        subaru
41141        subaru
41142        subaru
41143        subaru
Name: make, Length: 41144, dtype: category
Categories (136, str): ['am general', 'asc incorporated', 'acura', 'alfa romeo', ..., 'volvo', 'wallace environmental', 'yugo', 'smart']

In [24]:
ordered_make.cat.rename_categories(
    {c:c.lower() for c in ordered_make.cat.categories})

0        alfa romeo
1           ferrari
2             dodge
3             dodge
4            subaru
            ...    
41139        subaru
41140        subaru
41141        subaru
41142        subaru
41143        subaru
Name: make, Length: 41144, dtype: category
Categories (136, str): ['am general' < 'asc incorporated' < 'acura' < 'alfa romeo' ... 'volvo' < 'wallace environmental' < 'yugo' < 'smart']

In [25]:
ordered_make.cat.reorder_categories(
    sorted(cat_make.cat.categories, key=str.lower))

0        Alfa Romeo
1           Ferrari
2             Dodge
3             Dodge
4            Subaru
            ...    
41139        Subaru
41140        Subaru
41141        Subaru
41142        Subaru
41143        Subaru
Name: make, Length: 41144, dtype: category
Categories (136, str): ['Acura' < 'Alfa Romeo' < 'AM General' < 'American Motors Corporation' ... 'Volvo' < 'VPG' < 'Wallace Environmental' < 'Yugo']

#### Category Gotchas

In [26]:
(ordered_make
    .head(100)
    .value_counts()
)

make
Dodge                          17
Ford                            8
Oldsmobile                      8
Buick                           7
Chevrolet                       5
                               ..
Vixen Motor Company             0
Volga Associated Automobile     0
Wallace Environmental           0
Yugo                            0
smart                           0
Name: count, Length: 136, dtype: int64

In [27]:
(cat_make
    .head(100)
    .groupby(cat_make.head(100), observed=False)
    .first()
)

make
AM General                           <NA>
ASC Incorporated                     <NA>
Acura                                <NA>
Alfa Romeo                     Alfa Romeo
American Motors Corporation          <NA>
                                  ...    
Volkswagen                     Volkswagen
Volvo                               Volvo
Wallace Environmental                <NA>
Yugo                                 <NA>
smart                                <NA>
Name: make, Length: 136, dtype: category
Categories (136, string[pyarrow]): [AM General, ASC Incorporated, Acura, Alfa Romeo, ..., Volvo, Wallace Environmental, Yugo, smart]

In [28]:
(make
    .head(100)
    .groupby(make.head(100))
    .first()
)

make
Alfa Romeo          Alfa Romeo
Audi                      Audi
BMW                        BMW
Buick                    Buick
CX Automotive    CX Automotive
                     ...      
Rolls-Royce        Rolls-Royce
Subaru                  Subaru
Toyota                  Toyota
Volkswagen          Volkswagen
Volvo                    Volvo
Name: make, Length: 25, dtype: string[pyarrow]

In [29]:
(cat_make
    .head(100)
    .groupby(cat_make.head(100), observed=True)
    .first()
)

make
Alfa Romeo          Alfa Romeo
Audi                      Audi
BMW                        BMW
Buick                    Buick
CX Automotive    CX Automotive
                     ...      
Rolls-Royce        Rolls-Royce
Subaru                  Subaru
Toyota                  Toyota
Volkswagen          Volkswagen
Volvo                    Volvo
Name: make, Length: 25, dtype: category
Categories (136, string[pyarrow]): [AM General, ASC Incorporated, Acura, Alfa Romeo, ..., Volvo, Wallace Environmental, Yugo, smart]

In [30]:
ordered_make.iloc[0]

'Alfa Romeo'

In [31]:
ordered_make.iloc[[0]]

0    Alfa Romeo
Name: make, dtype: category
Categories (136, str): ['AM General' < 'ASC Incorporated' < 'Acura' < 'Alfa Romeo' ... 'Volvo' < 'Wallace Environmental' < 'Yugo' < 'smart']

#### Generalization

In [32]:
def generalize_topn(ser, n=5, other='Other'):
    topn = ser.value_counts().index[:n]
    if isinstance(ser.dtype, pd.CategoricalDtype):
        ser = ser.cat.set_categories(
            topn.set_categories(list(topn) + [other]))
    return ser.where(ser.isin(topn), other)

In [33]:
cat_make.pipe(generalize_topn, n=20, other='NA')

0            NA
1            NA
2         Dodge
3         Dodge
4        Subaru
          ...  
41139    Subaru
41140    Subaru
41141    Subaru
41142    Subaru
41143    Subaru
Name: make, Length: 41144, dtype: category
Categories (21, str): ['Chevrolet', 'Ford', 'Dodge', 'GMC', ..., 'Volvo', 'Hyundai', 'Chrysler', 'NA']

In [34]:
def generalize_mapping(ser, mapping, default):
    seen = None
    res = ser.astype(str)
    for old, new in mapping.items():
        mask = ser.str.contains(old)
        if seen is None:
            seen = mask
        else:
            seen |= mask
        res = res.where(~mask, new)
    res = res.where(seen, default)
    return res.astype('category')

In [35]:
generalize_mapping(cat_make, {
    'Ford':'US', 'Tesla':'US',
    'Chevrolet':'US', 'Dodge':'US',
    'Oldsmobile':'US', 'Plymouth':'US',
    'BMW':'German'}, 'Other')

0        Other
1        Other
2           US
3           US
4        Other
         ...  
41139    Other
41140    Other
41141    Other
41142    Other
41143    Other
Name: make, Length: 41144, dtype: category
Categories (3, str): ['German', 'Other', 'US']

# Dataframes

#### A Simple Python Version

In [36]:
df = {
    'index': [0, 1, 2],
    'cols': [
        {'name':'growth',
         'data': [.5, .7, 1.2]},
        {'name':'Name',
         'data':['Paul', 'George', 'Ringo']},
    ]
}

In [37]:
def get_row(df, idx):
    results = []
    value_idx = df['index'].index(idx)
    for col in df['cols']:
        results.append(col['data'][value_idx])
    return results

In [38]:
get_row(df, 1)

[0.7, 'George']

In [39]:
def get_col(df, name):
    for col in df['cols']:
        if col['name'] == name:
            return col['data']

In [40]:
get_col(df, 'Name')

['Paul', 'George', 'Ringo']

#### Dataframes

In [41]:
df = pd.DataFrame({
    'growth': [.5, .7, 1.2],
    'Name': ['Paul', 'George', 'Ringo']})

In [42]:
print(df)

   growth    Name
0     0.5    Paul
1     0.7  George
2     1.2   Ringo


In [43]:
df.iloc[2]

growth      1.2
Name      Ringo
Name: 2, dtype: object

In [44]:
df['Name']

0      Paul
1    George
2     Ringo
Name: Name, dtype: str

In [45]:
type(df['Name'])

pandas.Series

In [46]:
df['Name'].str.lower()

0      paul
1    george
2     ringo
Name: Name, dtype: str

#### Construction

In [47]:
print(pd.DataFrame([
    {'growth':.5, 'Name': 'Paul'},
    {'growth':.7, 'Name': 'George'},
    {'growth':1.2, 'Name': 'Ringo'}]))

   growth    Name
0     0.5    Paul
1     0.7  George
2     1.2   Ringo


In [48]:
csv_file = StringIO("""growth, Name
    .5, Paul
    .7, George
    1.2, Ringo
""")

In [49]:
print(pd.read_csv(csv_file, dtype_backend='pyarrow', engine='pyarrow'))

   growth     Name
0     0.5     Paul
1     0.7   George
2     1.2    Ringo


In [50]:
np.random.seed(42)

print(pd.DataFrame(np.random.randn(10, 3),
                  columns=['a', 'b', 'c']))

          a         b         c
0  0.496714 -0.138264  0.647689
1  1.523030 -0.234153 -0.234137
2  1.579213  0.767435 -0.469474
3  0.542560 -0.463418 -0.465730
4  0.241962 -1.913280 -1.724918
5 -0.562288 -1.012831  0.314247
6 -0.908024 -1.412304  1.465649
7 -0.225776  0.067528 -1.424748
8 -0.544383  0.110923 -1.150994
9  0.375698 -0.600639 -0.291694


#### Dataframe Axis

In [52]:
df.axes

[RangeIndex(start=0, stop=3, step=1), Index(['growth', 'Name'], dtype='str')]

In [53]:
df.sum(axis=0)

growth                2.4
Name      PaulGeorgeRingo
dtype: object

In [54]:
df.sum(axis=1)

TypeError: unsupported operand type(s) for +: 'float' and 'str'

In [55]:
df.sum(axis='index')

growth                2.4
Name      PaulGeorgeRingo
dtype: object

In [56]:
# df.sum(axis=1)

In [57]:
df.axes[0]

RangeIndex(start=0, stop=3, step=1)

In [58]:
df.axes[1]

Index(['growth', 'Name'], dtype='str')

In [60]:
df = pd.DataFrame({'Score1': [None, None],
                   'Score2': [85, 90]})

In [61]:
print(df)

  Score1  Score2
0   None      85
1   None      90


In [62]:
df.apply(np.sum, axis=0)

Score1      0
Score2    175
dtype: int64

# Similarities with Series & DataFrames

### Getting Data

In [77]:
url = 'https://en.wikipedia.org/wiki/'\
'Historical_rankings_of_presidents_of_the_United_States'

In [78]:
# pres_dfs = pd.read_html(url, dtype_backend='pyarrow')

In [79]:
headers = {

    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"

}

response = requests.get(url, headers=headers)

response.raise_for_status()

In [80]:
pres_dfs = pd.read_html(StringIO(response.text), dtype_backend="pyarrow")

In [81]:
pres_dfs

[             Seq. [b][c]             President        Political party  \
 0                      1     George Washington            Independent   
 1                      2            John Adams             Federalist   
 2                      3      Thomas Jefferson  Democratic-Republican   
 3                      4         James Madison  Democratic-Republican   
 4                      5          James Monroe  Democratic-Republican   
 ..                   ...                   ...                    ...   
 41                    43        George W. Bush             Republican   
 42                    44          Barack Obama             Democratic   
 43              45/47[b]          Donald Trump             Republican   
 44                    46             Joe Biden             Democratic   
 45  Total surveyed[b][c]  Total surveyed[b][c]   Total surveyed[b][c]   
 
     APSA 2024[27][26]  Siena 2022[32] C-SPAN 2021[20] Siena 2018[33]  \
 0                   3               

In [82]:
df = pres_dfs[2]

In [83]:
df

,Rank,Liberals (n = 190),Conservatives (n = 50)
0,1,Abraham Lincoln,Abraham Lincoln
1,2,Franklin D. Roosevelt,George Washington
2,3,George Washington,Franklin D. Roosevelt
3,4,Thomas Jefferson,Thomas Jefferson
4,5,Theodore Roosevelt,Theodore Roosevelt
...,...,...,...
13,32,James Buchanan,Franklin Pierce
14,33,Andrew Johnson,Andrew Johnson
15,34,Ulysses S. Grant,James Buchanan
16,35,Richard Nixon,Ulysses S. Grant


In [85]:
for i, table in enumerate(pres_dfs):
    print(i, table.columns)

0 Index(['Seq. [b][c]', 'President', 'Political party', 'APSA 2024[27][26]',
       'Siena 2022[32]', 'C-SPAN 2021[20]', 'Siena 2018[33]', 'APSA 2018[26]',
       'C-SPAN 2017[17]', 'PHN 2016[24]', 'APSA 2015[23]', 'USPC 2011[34]',
       'Siena 2010[35][36]', 'C-SPAN 2009[37]', 'Times 2008[38]',
       'WSJ 2005[13]', 'Siena 2002', 'WSJ 2000', 'C-SPAN 2000',
       'Schl. 1996[5]', 'R-McI 1996[39]', 'Siena 1994', 'Siena 1990',
       'Siena 1982', 'CT 1982', 'M-B 1982', 'Schl. 1962[4]', 'Schl. 1948'],
      dtype='str')
1 Index([0, 1], dtype='int64')
2 Index(['Rank', 'Liberals (n = 190)', 'Conservatives (n = 50)'], dtype='str')
3 Index(['Seq.', 'President', 'Political party', 'Bg', 'PL', 'CAb', 'RC', 'CAp',
       'HE', 'L', 'AC', 'WR', 'EAp', 'OA', 'Im', 'DA', 'Int', 'EAb', 'FPA',
       'LA', 'IQ', 'AM', 'EV', 'O'],
      dtype='str')
4 Index(['Seq.', 'President', 'Political party', 'VSA', 'DL', 'FPL', 'MA', 'HL',
       'O'],
      dtype='str')
5 Index(['Seq.', 'President', 'Politi

In [86]:
df = pres_dfs[0]  

In [87]:
# (df
#     .iloc[1:-1]
#     .rename(columns={'Political party': 'Party'})
#     .assign(Party=lambda df_:df_
#         .Party
#         .str.replace(r'\[.*\]', '')
#         .astype('category'))
# )

In [88]:
(df
    .iloc[1:-1]
    .rename(columns={'Political party': 'Party'})
    .assign(Party=lambda df_: df_['Party']
        .str.replace(r'\[.*?\]', '', regex=True)
        .astype('category'))
)

,Seq. [b][c],President,Party,APSA 2024[27][26],Siena 2022[32],C-SPAN 2021[20],Siena 2018[33],APSA 2018[26],C-SPAN 2017[17],PHN 2016[24],...,C-SPAN 2000,Schl. 1996[5],R-McI 1996[39],Siena 1994,Siena 1990,Siena 1982,CT 1982,M-B 1982,Schl. 1962[4],Schl. 1948
1,2,John Adams,Federalist,13,16,15,14,14,19,10,...,16,11,14,12,14,10,15,9,10,9
2,3,Thomas Jefferson,Democratic-Republican,5,5,7,5,5,7,5,...,7,4,4,5,3,2,5,4,5,5
3,4,James Madison,Democratic-Republican,11,10,16,7,12,17,15,...,18,17,10,9,8,9,17,14,12,14
4,5,James Monroe,Democratic-Republican,18,12,12,8,18,13,14,...,14,15,13,15,11,15,16,15,18,12
5,6,John Quincy Adams,Democratic-Republican,20,17,17,18,23,21,17,...,19,18,18,17,16,17,19,16,13,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40,42,Bill Clinton,Democratic,12,14,19,15,13,15,19,...,21,20,23,16,–,–,–,–,–,–
41,43,George W. Bush,Republican,32,35,29,33,30,33,34,...,–,–,–,–,–,–,–,–,–,–
42,44,Barack Obama,Democratic,7,11,10,17,8,12,7,...,–,–,–,–,–,–,–,–,–,–
43,45/47[b],Donald Trump,Republican,45,43,41,42,44,–,–,...,–,–,–,–,–,–,–,–,–,–


In [89]:
df.dtypes

Seq. [b][c]          string[pyarrow]
President            string[pyarrow]
Political party      string[pyarrow]
APSA 2024[27][26]     int64[pyarrow]
Siena 2022[32]        int64[pyarrow]
                          ...       
Siena 1982           string[pyarrow]
CT 1982              string[pyarrow]
M-B 1982             string[pyarrow]
Schl. 1962[4]        string[pyarrow]
Schl. 1948           string[pyarrow]
Length: 28, dtype: object

In [ ]:
#### Part II